In [ ]:
import pynucastro as pyna
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib widget


Reaction rates from files

In [ ]:
rates_reaclib=pyna.rates.library.Library(libfile='Nuclear_Data/run1/Reaclib_18_9_20')
rates_reaclib_exp=pyna.rates.library.Library(libfile='Nuclear_Data/run3/Reaclib_exp')
rates_beta_reddi=pd.read_csv('Empirical_formulas/Alpha_empirical/Reddi/Empirical_beta_decay_reddi.csv')
rates_weak_tabular= pyna.TabularLibrary(ordering=["ffn", "oda", "pruet_fuller", "langanke", "suzuki"]).get_rates()
sources_exp=list(pd.read_csv('Nuclear_Data/run3/sources_exp')['0'])
ffn_rates = [r for r in rates_weak_tabular if r.rfile.startswith("ffn")]
suzuki_rates = [r for r in rates_weak_tabular if r.rfile.startswith("suzuki")]
pruet_rates = [r for r in rates_weak_tabular if r.rfile.startswith("pruet")]
langanke_rates = [r for r in rates_weak_tabular if r.rfile.startswith("langanke")]
oda_rates = [r for r in rates_weak_tabular if r.rfile.startswith("oda")]

Filtros

In [ ]:
filter_alpha=pyna.RateFilter(
    products=['he4'],
    exact=False,
    max_reactants=1,
    max_products=2,
    filter_function=lambda r: r.reactants[0].Z==r.products[0].Z+r.products[1].Z and r.Q>0)

filter_alpha_teo=pyna.RateFilter(
    products=['he4'],
    exact=False,
    max_reactants=1,
    max_products=2,
    filter_function=lambda r: r.reactants[0].Z==r.products[0].Z+r.products[1].Z and r.Q>0 and r.source['Label'] not in sources_exp )

filter_beta_minus=pyna.RateFilter(
    max_reactants=1,
    max_products=1,
    filter_function=lambda r: r.reactants[0].Z==r.products[0].Z-1 and r.Q>0)

filter_beta_minus_teo=pyna.RateFilter(
    max_reactants=1,
    max_products=1,
    filter_function=lambda r: r.reactants[0].Z==r.products[0].Z-1 and r.Q>0 and r.source['Label'] not in sources_exp)


rates_beta_reaclib=rates_reaclib.filter(filter_beta_minus_teo)
rates_alpha_reaclib=rates_reaclib.filter(filter_alpha_teo)
rates_beta_exp=rates_reaclib_exp.filter(filter_beta_minus)
rates_alpha_exp=rates_reaclib_exp.filter(filter_alpha)

In [ ]:
#Reaclib
N_b_reaclib=[]
Z_b_reaclib=[]
lamba_b_reaclib=[]
q_b_reaclib=[]
for rate in rates_beta_reaclib.get_rates():
    N_b_reaclib.append(rate.reactants[0].N)
    Z_b_reaclib.append(rate.reactants[0].Z)
    lamba_b_reaclib.append(rate.eval(1e9))
    q_b_reaclib.append(rate.Q)

N_a_reaclib=[]
Z_a_reaclib=[]
lamba_a_reaclib=[]
q_a_reaclib=[]
for rate in rates_alpha_reaclib.get_rates():
    N_a_reaclib.append(rate.reactants[0].N)
    Z_a_reaclib.append(rate.reactants[0].Z)
    lamba_a_reaclib.append(rate.eval(1e9))
    q_a_reaclib.append(rate.Q)
#Reaclib experimental
N_b_exp=[]
Z_b_exp=[]
lamba_b_exp=[]
q_b_exp=[]
for rate in rates_beta_exp.get_rates():
    N_b_exp.append(rate.reactants[0].N)
    Z_b_exp.append(rate.reactants[0].Z)
    lamba_b_exp.append(rate.eval(1e9))
    q_b_exp.append(rate.Q)

N_a_exp=[]
Z_a_exp=[]
lamba_a_exp=[]
q_a_exp=[]
for rate in rates_alpha_exp.get_rates():
    N_a_exp.append(rate.reactants[0].N)
    Z_a_exp.append(rate.reactants[0].Z)
    lamba_a_exp.append(rate.eval(1e9))
    q_a_exp.append(rate.Q)
#Raddi




In [ ]:
high_Z = 50 #max(r.products[0].Z for r in rates_weak_tabular)
high_N = 100#max(r.products[0].N for r in rates_weak_tabular)
#max_size = 5 * (max(high_Z, high_N) // 5 + 1)
fig, ax = plt.subplots()

#ax.scatter([r.reactants[0].N for r in langanke_rates if r.weak_type.startswith("beta")],
#           [r.reactants[0].Z for r in langanke_rates if r.weak_type.startswith("beta")],
#           marker="s", color="C2", label="Langanke")

ax.scatter([r.reactants[0].N for r in rates_beta_exp.get_rates()],
           [r.reactants[0].Z for r in rates_beta_exp.get_rates()],
           marker="s", color="C0", label="Experiments")


ax.legend()
ax.set_xlabel("N")
ax.set_ylabel("Z")
ax.set_xlim(-0.5, 166.5)
ax.set_ylim(-0.5, 106.5)
ax.set_aspect("equal")
ax.grid()
ax.set_title(r"$\beta$-decay")
fig.set_size_inches(8, 8)

In [ ]:
high_Z = 50 #max(r.products[0].Z for r in rates_weak_tabular)
high_N = 100#max(r.products[0].N for r in rates_weak_tabular)
#max_size = 5 * (max(high_Z, high_N) // 5 + 1)
fig, ax = plt.subplots()

ax.scatter([r.reactants[0].N for r in rates_beta_reaclib.get_rates()],
           [r.reactants[0].Z for r in rates_beta_reaclib.get_rates()],
           marker="s", color="C1", label="Reaclib_Theoretical")
ax.scatter([r.reactants[0].N for r in langanke_rates if r.weak_type.startswith("beta")],
           [r.reactants[0].Z for r in langanke_rates if r.weak_type.startswith("beta")],
           marker="s", color="C2", label="Langanke")

ax.legend()
ax.set_xlabel("N")
ax.set_ylabel("Z")
#ax.set_xlim(-0.5, 236.5)
#ax.set_ylim(-0.5, 110.5)
ax.set_aspect("equal")
ax.grid()
ax.set_title(r"$\beta$-decay")
fig.set_size_inches(14, 8)

In [ ]:
high_Z = 50 #max(r.products[0].Z for r in rates_weak_tabular)
high_N = 100#max(r.products[0].N for r in rates_weak_tabular)
#max_size = 5 * (max(high_Z, high_N) // 5 + 1)
fig, ax = plt.subplots()


ax.scatter([r.reactants[0].N for r in rates_beta_reaclib.get_rates()],
           [r.reactants[0].Z for r in rates_beta_reaclib.get_rates()],
           marker="s", color="C1", label="Reaclib_Theoretical")
ax.scatter([r.reactants[0].N for r in langanke_rates if r.weak_type.startswith("beta")],
           [r.reactants[0].Z for r in langanke_rates if r.weak_type.startswith("beta")],
           marker="s", color="C2", label="Langanke")
ax.scatter([r.reactants[0].N for r in rates_beta_exp.get_rates()],
           [r.reactants[0].Z for r in rates_beta_exp.get_rates()],
           marker="s", color="C0", label="Experiments")

ax.legend()
ax.set_xlabel("N")
ax.set_ylabel("Z")
#ax.set_xlim(-0.5, 236.5)
#ax.set_ylim(-0.5, 110.5)
ax.set_aspect("equal")
ax.grid()
ax.set_title(r"$\beta$-decay")
fig.set_size_inches(14, 8)

In [ ]:
fig, ax = plt.subplots()


ax.scatter([rates_beta_raddi['N']],
           [rates_beta_raddi['Z']],
           marker="s", color="C7", label="Reddi")
ax.scatter([r.reactants[0].N for r in ffn_rates if r.weak_type.startswith("beta")],
           [r.reactants[0].Z for r in ffn_rates if r.weak_type.startswith("beta")],
           marker="s", color="C6", label="FFN")
ax.scatter([r.reactants[0].N for r in langanke_rates if r.weak_type.startswith("beta")],
           [r.reactants[0].Z for r in langanke_rates if r.weak_type.startswith("beta")],
           marker="s", color="C2", label="Langanke")
ax.scatter([r.reactants[0].N for r in oda_rates if r.weak_type.startswith("beta")],
           [r.reactants[0].Z for r in oda_rates if r.weak_type.startswith("beta")],
           marker="s", color="C4", label="Oda")
ax.scatter([r.reactants[0].N for r in pruet_rates if r.weak_type.startswith("beta")],
           [r.reactants[0].Z for r in pruet_rates if r.weak_type.startswith("beta")],
           marker="s", color="C5", label="Pruet & Fuller")
ax.scatter([r.reactants[0].N for r in suzuki_rates if r.weak_type.startswith("beta")],
           [r.reactants[0].Z for r in suzuki_rates if r.weak_type.startswith("beta")],
           marker="s", color="C3", label="Suzuki")
ax.scatter([r.reactants[0].N for r in rates_beta_exp.get_rates()],
           [r.reactants[0].Z for r in rates_beta_exp.get_rates()],
           marker="s", color="C0", label="Experiments")

ax.legend()
ax.set_xlabel("N")
ax.set_ylabel("Z")
ax.set_xlim(-0.5, 236.5)
ax.set_ylim(-0.5, 110.5)
ax.set_aspect("equal")
ax.grid()
ax.set_title(r"$\beta$-decay")
fig.set_size_inches(8, 8)

In [ ]:
print(len(rates_beta_reaclib.get_rates()),len(rates_alpha_reaclib.get_rates()))
print(len(rates_beta_exp.get_rates()),len(rates_alpha_exp.get_rates()))

fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.scatter(N_a_exp,Z_a_exp,np.log(lamba_a_exp),label='Experimental alpha',s=0.8,color='red')
ax.scatter(N_a_reaclib,Z_a_reaclib,np.log(lamba_a_reaclib),label='reaclib alpha',s=0.8,color='blue')
#ax.scatter(N_vioa,Z_vioa,np.log(lambda_vioa),label='Viola',s=1, marker='X',color='green')
plt.legend()
ax.set_zlabel('log(Decay rate (1/s))')
plt.ylabel('Z of reactant')
plt.xlabel('N of reactant')
plt.show()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.scatter(N_b_exp,Z_b_exp,np.log(lamba_b_exp),label='Experimental beta',s=0.8,color='red')
ax.scatter(N_b_reaclib,Z_b_reaclib,np.log(lamba_b_reaclib),label='reaclib beta',s=0.8,color='blue')
ax.scatter(rates_beta_raddi['N'],rates_beta_raddi['Z'],np.log(rates_beta_raddi['decay rate (1/s)']),s=0.8,color='green')
plt.legend()
ax.set_zlabel('log(Decay rate (1/s))')
plt.ylabel('Z of reactant')
ax.set_zlim(-40,10)
plt.xlabel('N of reactant')
plt.show()